# ReadAnyDoc in 2 minutes

Parse a PDF to Markdown and extract typed fields with a citation for every value. Open source, runs on your own Anthropic key (about 1 cent for this notebook).

[GitHub](https://github.com/arpitpatel25/ReadAnyDoc) · [Docs](https://arpitpatel25.github.io/ReadAnyDoc/docs/)

## 1. Install

The `engine` extra adds a small layout model and OCR that run on the CPU. The first run downloads the layout weights once.

In [ ]:
!pip install -q "readanydoc[engine]"
!apt-get -qq install -y tesseract-ocr > /dev/null

## 2. Your Anthropic key

Add `ANTHROPIC_API_KEY` in Colab's **Secrets** panel (the key icon on the left), or paste it when asked. It is not printed or saved in the notebook.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")

## 3. A real document

The US government's sample mortgage Loan Estimate (public domain). Use `files.upload()` instead to try your own.

In [ ]:
!wget -q https://raw.githubusercontent.com/arpitpatel25/ReadAnyDoc/main/examples/showcase/loan_estimate.pdf
# from google.colab import files; files.upload()

## 4. Parse to Markdown

In [ ]:
import readanydoc

result = readanydoc.parse("loan_estimate.pdf")
print(result.markdown[:2000])
print("cost $", result.usage.cost_usd, "| warnings:", result.warnings)

## 5. Extract typed fields, with citations

In [ ]:
from typing import Optional
from pydantic import BaseModel

class LoanEstimate(BaseModel):
    lender: str
    applicants: list[str]
    loan_amount: Optional[float]
    interest_rate_percent: Optional[float]
    monthly_principal_and_interest: Optional[float]
    has_prepayment_penalty: Optional[bool]
    estimated_cash_to_close: Optional[float]

out = readanydoc.extract("loan_estimate.pdf", schema=LoanEstimate)
out.data

In [ ]:
for field, cites in out.citations.items():
    c = cites[0]
    print(f"{field:32} {c.content[:40]!r:44} page {c.bbox.page if c.bbox else '?'}  {c.confidence}")

## Did it work?

If ReadAnyDoc saved you time, **[star it on GitHub](https://github.com/arpitpatel25/ReadAnyDoc)**. It is the quickest way to help other developers find it.

### Next

- [Use cases](https://arpitpatel25.github.io/ReadAnyDoc/docs/use-cases/) with a schema for invoices, statements, receipts, tax forms, contracts
- [Use it inside Claude or Cursor (MCP)](https://arpitpatel25.github.io/ReadAnyDoc/docs/get-started/mcp/)
- Questions: [GitHub Discussions](https://github.com/arpitpatel25/ReadAnyDoc/discussions)